# MLOps Zoomcamp Monitoring Homework


In [1]:
import pandas as pd
import joblib
from evidently.report import Report
from evidently import ColumnMapping
from evidently.metrics import ColumnSummaryMetric, ColumnQuantileMetric


In [2]:
# Load March data + reference
march_data = pd.read_parquet('../data/green_tripdata_2024-03.parquet')
reference_data = pd.read_parquet('../data/reference.parquet')

In [3]:
print(f"Q1:What is the shape of the downloaded data?{march_data.shape}")

Q1:What is the shape of the downloaded data?(57457, 20)


In [4]:
march_data.describe()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
count,57457.000000,57457,57457,55360.000000,57457.000000,57457.000000,55360.000000,57457.000000,57457.000000,57457.000000,57457.000000,57457.000000,57457.000000,0.0,57457.000000,57457.000000,55360.000000,55353.000000,55360.000000
mean,1.877334,2024-03-16 04:02:52.405399,2024-03-16 04:21:00.076039,1.179986,95.524688,138.629149,1.309538,13.522828,17.313474,0.904472,0.577410,2.386255,0.192537,NaN,0.979378,22.904832,1.321062,1.038047,0.737730
min,1.000000,2008-12-31 23:02:24,2008-12-31 23:02:30,1.000000,1.000000,1.000000,0.000000,0.000000,-295.080000,-2.500000,-0.500000,-1.560000,0.000000,NaN,-1.000000,-296.080000,1.000000,1.000000,-2.750000
25%,2.000000,2024-03-08 13:53:56,2024-03-08 14:13:49,1.000000,74.000000,74.000000,1.000000,1.100000,9.300000,0.000000,0.500000,0.000000,0.000000,NaN,1.000000,13.440000,1.000000,1.000000,0.000000
50%,2.000000,2024-03-15 22:49:01,2024-03-15 23:09:52,1.000000,75.000000,138.000000,1.000000,1.790000,13.500000,0.000000,0.500000,2.000000,0.000000,NaN,1.000000,18.500000,1.000000,1.000000,0.000000
75%,2.000000,2024-03-23 20:11:25,2024-03-23 20:34:48,1.000000,97.000000,220.000000,1.000000,3.100000,19.800000,1.000000,0.500000,3.610000,0.000000,NaN,1.000000,27.050000,2.000000,1.000000,2.750000
max,2.000000,2024-04-01 00:01:45,2024-04-01 16:11:00,99.000000,265.000000,265.000000,9.000000,125112.200000,841.600000,10.000000,4.250000,150.000000,26.760000,NaN,1.000000,856.980000,5.000000,2.000000,2.750000
std,0.328056,NaN,NaN,1.356719,57.285088,76.295346,0.967749,770.416255,14.958249,1.382446,0.366916,3.159273,1.184551,NaN,0.154253,17.013735,0.497858,0.191311,1.218039


In [5]:
# Load model
with open('../models/lin_reg.bin', 'rb') as f_in:
    model = joblib.load(f_in)

In [6]:
# Features
num_features = ["passenger_count", "trip_distance", "fare_amount", "total_amount"]
cat_features = ["PULocationID", "DOLocationID"]

In [7]:
# remove NaN value
march_data = march_data.fillna(0)

In [8]:
# Strictly filter March 2024
march_data = march_data[
    (march_data['lpep_pickup_datetime'].dt.year == 2024) &
    (march_data['lpep_pickup_datetime'].dt.month == 3)
].copy()

In [9]:
# Predict
march_data['prediction'] = model.predict(march_data[num_features + cat_features].fillna(0))
march_data['date'] = march_data['lpep_pickup_datetime'].dt.date

### Evidently Report

In [10]:
march_data['date'] = march_data['lpep_pickup_datetime'].dt.date

In [11]:
# Column mapping
column_mapping = ColumnMapping(
    prediction='prediction',
    numerical_features=num_features,
    categorical_features=cat_features,
    target=None
)

In [15]:
daily_results = []

for date, group in march_data.groupby('date'):
    report = Report(metrics=[
        ColumnQuantileMetric(column_name='fare_amount', quantile=0.5)
    ])
    report.run(reference_data=reference_data, current_data=group, column_mapping=column_mapping)
    result = report.as_dict()
    median_value = result['metrics'][0]['result']['current']['value']
    daily_results.append({'date': date, 'median_fare': median_value})
    print(f"{date}: median_fare={median_value}")

2024-03-01: median_fare=13.5
2024-03-02: median_fare=13.5
2024-03-03: median_fare=14.2
2024-03-04: median_fare=12.8
2024-03-05: median_fare=13.5
2024-03-06: median_fare=12.8


2024-03-07: median_fare=13.5
2024-03-08: median_fare=13.5
2024-03-09: median_fare=13.5
2024-03-10: median_fare=14.2
2024-03-11: median_fare=12.8
2024-03-12: median_fare=13.5
2024-03-13: median_fare=13.5


2024-03-14: median_fare=14.2
2024-03-15: median_fare=13.5
2024-03-16: median_fare=14.2
2024-03-17: median_fare=13.5
2024-03-18: median_fare=13.5
2024-03-19: median_fare=13.5


2024-03-20: median_fare=12.8
2024-03-21: median_fare=13.5
2024-03-22: median_fare=13.5
2024-03-23: median_fare=12.8
2024-03-24: median_fare=14.2
2024-03-25: median_fare=13.5


2024-03-26: median_fare=13.5
2024-03-27: median_fare=13.5
2024-03-28: median_fare=13.5
2024-03-29: median_fare=13.5
2024-03-30: median_fare=14.2
2024-03-31: median_fare=13.5


In [16]:
# Find max median
max_median = max(r['median_fare'] for r in daily_results)
print(f"\n Max daily median fare for March 2024: {max_median}")


 Max daily median fare for March 2024: 14.2


In [17]:
# Optional: convert to DataFrame for display
df_daily_results = pd.DataFrame(daily_results)
df_daily_results


,date,median_fare
0,2024-03-01,13.5
1,2024-03-02,13.5
2,2024-03-03,14.2
3,2024-03-04,12.8
4,2024-03-05,13.5
5,2024-03-06,12.8
6,2024-03-07,13.5
7,2024-03-08,13.5
8,2024-03-09,13.5
9,2024-03-10,14.2
